# Valence: treino do classificador de doenças em folhas

Este notebook treina o modelo de visão computacional do serviço de IA do Valence. O modelo recebe a foto de uma folha de **milho, tomate ou batata** e indica a doença mais provável entre **17 classes** (incluindo folha saudável). A API Java usa essa resposta na triagem da ocorrência enviada pelo produtor.

**Como executar:** no Google Colab, *Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4* e depois *Executar tudo*. Os checkpoints ficam no Google Drive, então se o Colab desconectar basta executar tudo de novo: o treino continua de onde parou.

| Seção | Conteúdo |
| --- | --- |
| 0 | Configuração do ambiente |
| 1 | Dados do PlantVillage e divisão treino, validação e teste |
| 2 | Pré-processamento e aumento de dados |
| 3 | Experimento 1: MobileNetV3-Large x EfficientNet-B0 |
| 4 | Comparação no teste e escolha da arquitetura |
| 5 | Calibração da confiança |
| 6 | Experimento 2: fotos de campo do PlantDoc, sem ajuste |
| 7 | Experimento 3: ajuste fino com fotos de campo |
| 8 | Grad-CAM: para onde o modelo olha |
| 9 | Exportação do modelo e das evidências |
| 10 | Limitações observadas |

## 0. Configuração

Três decisões aqui:

- **Seed fixa (42)** em Python, NumPy e PyTorch, para que a divisão dos dados e a inicialização sejam as mesmas a cada execução. Na GPU algumas operações continuam não determinísticas, então os números podem variar na terceira casa decimal.
- **Google Drive montado** para guardar checkpoints a cada época. O Colab gratuito desconecta; sem isso perderíamos o treino.
- **`MODO_RAPIDO`** usa poucas imagens por classe e uma época por fase. Serve só para conferir que o notebook roda do início ao fim, inclusive em CPU. Os números do modo rápido não têm valor como avaliação.

In [ ]:
import sys

EM_COLAB = "google.colab" in sys.modules
if EM_COLAB:
    %pip install -q "datasets==5.2.0" "transformers==5.22.2"

In [ ]:
# Bibliotecas
import copy
import hashlib
import json
import os
import random
import tempfile
import time
import urllib.request
import zipfile
from datetime import UTC, datetime
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision
from huggingface_hub import DatasetCard, HfFileSystem, hf_hub_download
from IPython.display import Markdown, display
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms

# Modo de execução
MODO_RAPIDO = False
if os.environ.get("MODO_RAPIDO") == "1":
    MODO_RAPIDO = True

SEED = 42


def fixar_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


fixar_seed(SEED)

DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USAR_AMP = DISPOSITIVO.type == "cuda"
torch.backends.cudnn.benchmark = True

print(f"Modo rápido: {MODO_RAPIDO}")
print(
    f"PyTorch {torch.__version__}, torchvision {torchvision.__version__}, datasets "
    f"{datasets.__version__}"
)
if DISPOSITIVO.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Sem GPU: o treino completo fica muito lento. No Colab, escolha GPU T4.")

In [ ]:
# Pastas de trabalho
if EM_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    SAIDA = Path("/content/drive/MyDrive/valence-ia/treino_visao")
    DADOS = Path("/content/dados")
else:
    base = Path(
        os.environ.get("VALENCE_SAIDA", Path(tempfile.gettempdir()) / "valence-treino-visao")
    )
    SAIDA = base / "saida"
    DADOS = base / "dados"

CHECKPOINTS = SAIDA / "checkpoints"
MODELOS = SAIDA / "modelos"
AVALIACAO = SAIDA / "avaliacao" / "visao"
for pasta in (CHECKPOINTS, MODELOS, AVALIACAO, DADOS):
    pasta.mkdir(parents=True, exist_ok=True)

# Hiperparâmetros
CONFIG = {
    "tamanho_entrada": 224,
    "redimensionar": 256,
    "lote": 16 if MODO_RAPIDO else 64,
    "epocas_congelado": 1 if MODO_RAPIDO else 2,
    "epocas_descongelado": 1 if MODO_RAPIDO else 6,
    "lr_cabeca": 1e-3,
    "lr_ajuste": 1e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.1,
    "paciencia": 2,
    "fracao_validacao": 0.15,
    "fracao_teste_plantdoc": 0.30,
    "fracao_plantdoc_no_lote": 0.30,
    "rapido_treino_por_classe": 20,
    "rapido_teste_por_classe": 8,
    "rapido_plantdoc_por_classe": 6,
    "rapido_plantdoc_outras": 30,
}
NUM_WORKERS = 2 if DISPOSITIVO.type == "cuda" else 0

print(f"Saída: {SAIDA}")
print(json.dumps(CONFIG, indent=2))

In [ ]:
# Funções auxiliares de gravação
def para_json(valor):
    if isinstance(valor, (np.integer,)):
        return int(valor)
    if isinstance(valor, (np.floating,)):
        return float(valor)
    if isinstance(valor, np.ndarray):
        return valor.tolist()
    if isinstance(valor, Path):
        return str(valor)
    raise TypeError(type(valor))


def salvar_json(nome: str, dados: dict) -> None:
    texto = json.dumps(dados, ensure_ascii=False, indent=2, default=para_json)
    (AVALIACAO / nome).write_text(texto, encoding="utf-8")


def salvar_figura(fig, nome: str) -> None:
    fig.savefig(AVALIACAO / nome, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)


RESULTADOS = {"modoRapido": MODO_RAPIDO, "seed": SEED}

## 1. Dados: PlantVillage

O PlantVillage (Mohanty, Hughes e Salathé, 2016) tem cerca de 54 mil fotos de folhas, tiradas em laboratório com fundo liso. Usamos a versão colorida e só as culturas do Valence: milho, tomate e batata.

**Por que não `datasets.load_dataset("mohanty/PlantVillage", "color")`:** o repositório `mohanty/PlantVillage` depende de um script de carga (`plant_village.py`), e a biblioteca `datasets` deixou de executar scripts de carga na versão 4.0. Com a versão atual a chamada falha com `BuilderConfig 'color' not found`. Por isso lemos os mesmos arquivos oficiais do repositório com o `huggingface_hub`: o `data.zip` com as imagens, os arquivos `splits/color_train.txt` e `splits/color_test.txt` (divisão oficial) e o `leaf_grouping/leaf-map.json`. O `leaf_id` é calculado com a mesma regra do script.

**Por que o `leaf_id` importa:** a mesma folha foi fotografada várias vezes. Se uma foto da folha cai no treino e outra no teste, o modelo "reconhece a folha" e a métrica fica otimista. A divisão oficial foi feita para separar treino e teste por folha. Conferimos isso: as folhas que aparecerem dos dois lados saem do treino, para o teste ficar limpo. Ao separar 15% do treino para validação, também agrupamos por folha.

Parte das imagens não tem `leaf_id` no mapa oficial; cada uma delas vira um grupo próprio. Para essas, a separação por folha não é garantida, e isso entra nas limitações.

As classes vêm de `features["label"].names`, lidas do cartão do dataset no Hugging Face, e não de uma lista digitada.

In [ ]:
# Classes do PlantVillage lidas do cartão do dataset
REPO_PV = "mohanty/PlantVillage"
cartao = DatasetCard.load(REPO_PV, repo_type="dataset")
configs = cartao.data.to_dict()["dataset_info"]
configs = configs if isinstance(configs, list) else [configs]
config_cor = next(c for c in configs if c.get("config_name") == "color")
rotulo_info = next(f for f in config_cor["features"] if f["name"] == "label")
nomes_por_indice = rotulo_info["dtype"]["class_label"]["names"]
features = datasets.Features(
    {
        "label": datasets.ClassLabel(
            names=[v for _, v in sorted(nomes_por_indice.items(), key=lambda kv: int(kv[0]))]
        )
    }
)

PREFIXOS = ("Corn_(maize)___", "Potato___", "Tomato___")
CLASSES_PV = [nome for nome in features["label"].names if nome.startswith(PREFIXOS)]
print(
    f"{len(features['label'].names)} classes no PlantVillage, {len(CLASSES_PV)} de milho, "
    "batata e tomate"
)
assert len(CLASSES_PV) == 17

In [ ]:
# Rótulos do Valence (dados/rotulos.json)
URL_ROTULOS = "https://raw.githubusercontent.com/Valence-Team/valence-iot/main/dados/rotulos.json"


def carregar_rotulos() -> list[dict]:
    for candidato in (Path("../dados/rotulos.json"), Path("dados/rotulos.json")):
        if candidato.exists():
            return json.loads(candidato.read_text(encoding="utf-8"))
    try:
        with urllib.request.urlopen(URL_ROTULOS, timeout=20) as resposta:
            return json.loads(resposta.read().decode("utf-8"))
    except OSError:
        if not EM_COLAB:
            raise
    from google.colab import files

    print("Envie o arquivo dados/rotulos.json do repositório.")
    enviado = files.upload()
    return json.loads(next(iter(enviado.values())).decode("utf-8"))


ROTULOS_JSON = carregar_rotulos()
POR_CLASSE_PV = {item["classePlantVillage"]: item for item in ROTULOS_JSON}
faltando = [c for c in CLASSES_PV if c not in POR_CLASSE_PV]
assert not faltando, f"Classes sem rótulo: {faltando}"

ROTULOS = [POR_CLASSE_PV[c]["rotulo"] for c in CLASSES_PV]
NOMES = [POR_CLASSE_PV[c]["nome"] for c in CLASSES_PV]
CULTURAS = np.array([POR_CLASSE_PV[c]["cultura"] for c in CLASSES_PV])
NUM_CLASSES = len(ROTULOS)
INDICE_DA_CLASSE_PV = {c: i for i, c in enumerate(CLASSES_PV)}
INDICE_DO_ROTULO = {r: i for i, r in enumerate(ROTULOS)}

pd.DataFrame(
    {
        "saída": range(NUM_CLASSES),
        "rótulo": ROTULOS,
        "nome": NOMES,
        "cultura": CULTURAS,
        "classe PlantVillage": CLASSES_PV,
    }
)

In [ ]:
# Divisão oficial e leaf_id
arquivo_treino = hf_hub_download(REPO_PV, "splits/color_train.txt", repo_type="dataset")
arquivo_teste = hf_hub_download(REPO_PV, "splits/color_test.txt", repo_type="dataset")
arquivo_folhas = hf_hub_download(REPO_PV, "leaf_grouping/leaf-map.json", repo_type="dataset")
MAPA_FOLHAS = json.loads(Path(arquivo_folhas).read_text(encoding="utf-8"))


def obter_leaf_id(caminho: str) -> str:
    classe, arquivo = caminho.split("/")[2], caminho.split("/")[3]
    ident = arquivo.replace("_final_masked", "")
    if "___" in ident:
        ident = ident.split("___")[-1]
    ident = ident.split("copy")[0]
    for extensao in (".jpg", ".JPG", ".png", ".PNG"):
        ident = ident.replace(extensao, "")
    ident = ident.strip()
    sugestoes = MAPA_FOLHAS.get(ident.lower().strip())
    if not sugestoes:
        return f"fallback_{ident}"
    if len(sugestoes) == 1:
        return sugestoes[0]
    for sugestao in sugestoes:
        if classe in sugestao:
            return sugestao
    return f"fallback_{ident}"


def ler_split(arquivo: str) -> pd.DataFrame:
    texto = Path(arquivo).read_text(encoding="utf-8")
    linhas = [linha.strip() for linha in texto.splitlines() if linha.strip()]
    registros = []
    for caminho in linhas:
        partes = caminho.split("/")
        if len(partes) < 4 or partes[2] not in INDICE_DA_CLASSE_PV:
            continue
        indice = INDICE_DA_CLASSE_PV[partes[2]]
        registros.append(
            {
                "caminho": caminho,
                "alvo": indice,
                "rotulo": ROTULOS[indice],
                "cultura": CULTURAS[indice],
                "leaf_id": obter_leaf_id(caminho),
            }
        )
    return pd.DataFrame(registros)


treino_oficial = ler_split(arquivo_treino)
teste_pv = ler_split(arquivo_teste)

folhas_em_comum = set(treino_oficial["leaf_id"]) & set(teste_pv["leaf_id"])
fallback = (pd.concat([treino_oficial, teste_pv])["leaf_id"].str.startswith("fallback_")).mean()
removidas_do_treino = int(treino_oficial["leaf_id"].isin(folhas_em_comum).sum())
treino_oficial = treino_oficial[~treino_oficial["leaf_id"].isin(folhas_em_comum)]
print(f"Teste oficial: {len(teste_pv)} imagens")
print(f"Folhas presentes no treino e no teste ao mesmo tempo: {len(folhas_em_comum)}")
print(
    f"Imagens dessas folhas retiradas do treino: {removidas_do_treino}; treino restante: "
    f"{len(treino_oficial)}"
)
print(f"Imagens sem leaf_id no mapa (cada uma vira um grupo próprio): {fallback:.1%}")

In [ ]:
# Validação separada por folha e amostra do modo rápido
def amostrar_por_classe(df: pd.DataFrame, n: int) -> pd.DataFrame:
    return df.sample(frac=1.0, random_state=SEED).groupby("alvo").head(n)


if MODO_RAPIDO:
    treino_oficial = amostrar_por_classe(treino_oficial, CONFIG["rapido_treino_por_classe"])
    teste_pv = amostrar_por_classe(teste_pv, CONFIG["rapido_teste_por_classe"])

indices_validacao = []
for _, grupo in treino_oficial.groupby("alvo"):
    divisor = GroupShuffleSplit(n_splits=1, test_size=CONFIG["fracao_validacao"], random_state=SEED)
    _, posicoes = next(divisor.split(grupo, groups=grupo["leaf_id"]))
    indices_validacao.extend(grupo.index[posicoes])

validacao_pv = treino_oficial.loc[indices_validacao].reset_index(drop=True)
treino_pv = treino_oficial.drop(index=indices_validacao).reset_index(drop=True)
teste_pv = teste_pv.reset_index(drop=True)

assert not set(treino_pv["leaf_id"]) & set(validacao_pv["leaf_id"])
print(
    f"Treino: {len(treino_pv)}  Validação: {len(validacao_pv)} "
    f"({len(validacao_pv) / len(treino_oficial):.1%} do treino oficial)  Teste: "
    f"{len(teste_pv)}"
)

In [ ]:
# Download das imagens
PASTA_PV = DADOS / "plantvillage"


def abrir_zip_pv() -> zipfile.ZipFile:
    if MODO_RAPIDO:
        return zipfile.ZipFile(
            HfFileSystem().open(f"datasets/{REPO_PV}/data.zip", "rb", block_size=2**20)
        )
    return zipfile.ZipFile(hf_hub_download(REPO_PV, "data.zip", repo_type="dataset"))


def arquivo_local(membro: str) -> Path:
    return PASTA_PV / (
        hashlib.sha1(membro.encode("utf-8")).hexdigest()[:20] + Path(membro).suffix.lower()
    )


def baixar_do_zip(membros: list[str]) -> None:
    PASTA_PV.mkdir(parents=True, exist_ok=True)
    faltando = [m for m in membros if not arquivo_local(m).exists()]
    if not faltando:
        return
    inicio = time.time()
    with abrir_zip_pv() as arquivo_zip:
        existentes = set(arquivo_zip.namelist())
        faltando = [m for m in faltando if m in existentes]
        faltando.sort(key=lambda m: arquivo_zip.getinfo(m).header_offset)
        for membro in faltando:
            arquivo_local(membro).write_bytes(arquivo_zip.read(membro))
    print(f"{len(faltando)} imagens extraídas em {time.time() - inicio:.0f} s")


baixar_do_zip(pd.concat([treino_pv, validacao_pv, teste_pv])["caminho"].tolist())

In [ ]:
# Contagem por classe
contagem = (
    pd.DataFrame(
        {
            "treino": treino_pv["rotulo"].value_counts(),
            "validação": validacao_pv["rotulo"].value_counts(),
            "teste": teste_pv["rotulo"].value_counts(),
        }
    )
    .reindex(ROTULOS)
    .fillna(0)
    .astype(int)
)
contagem["total"] = contagem.sum(axis=1)
display(contagem)

fig, ax = plt.subplots(figsize=(10, 6))
contagem[["treino", "validação", "teste"]].plot.barh(
    stacked=True, ax=ax, color=["#2e7d32", "#81c784", "#c8e6c9"]
)
ax.invert_yaxis()
ax.set_xlabel("imagens")
ax.set_title("PlantVillage: imagens por classe (milho, tomate e batata)")
salvar_figura(fig, "contagem_plantvillage.png")

maior, menor = contagem["treino"].idxmax(), contagem["treino"].idxmin()
razao = contagem.loc[maior, "treino"] / max(contagem.loc[menor, "treino"], 1)
display(
    Markdown(
        f"**Desbalanceamento:** a maior classe no treino é `{maior}` "
        f"({contagem.loc[maior, 'treino']} imagens) "
        f"e a menor é `{menor}` ({contagem.loc[menor, 'treino']}), uma razão de {razao:.1f} "
        "para 1. "
        "Sem correção, o modelo tende a favorecer as classes grandes, e a acurácia esconde o "
        "erro nas pequenas. "
        "Por isso usamos perda ponderada pelo inverso da frequência e acompanhamos o **F1 "
        "macro**, que dá o mesmo peso a cada classe."
    )
)
RESULTADOS["dados"] = {
    "plantvillage": {
        "treino": len(treino_pv),
        "validacao": len(validacao_pv),
        "teste": len(teste_pv),
    },
    "folhasEmComumTreinoTeste": len(folhas_em_comum),
    "imagensRetiradasDoTreino": removidas_do_treino,
    "fracaoSemLeafId": float(fallback),
    "razaoDesbalanceamento": float(razao),
}
salvar_json(
    "dados_plantvillage.json", {"contagem": contagem.to_dict(orient="index"), **RESULTADOS["dados"]}
)

## 2. Pré-processamento e aumento de dados

O modelo foi pré-treinado no ImageNet, então usamos a mesma normalização (média e desvio por canal do ImageNet) e entrada de 224 x 224.

No **treino**, cada imagem sofre transformações aleatórias para o modelo não decorar detalhes do PlantVillage e se aproximar de fotos de celular:

- `RandomResizedCrop(224, scale=(0.6, 1))`: recorta entre 60% e 100% da imagem, simulando distância e enquadramento diferentes;
- giros horizontal e vertical e rotação de até 20 graus: a folha pode estar em qualquer orientação;
- `ColorJitter`: variações de brilho, contraste, saturação e tom, como em fotos ao sol ou na sombra;
- `RandomGrayscale` com 5% de chance: obriga o modelo a usar forma e textura da lesão, não só a cor.

Na **validação e no teste**, nada aleatório: `Resize(256)` e `CenterCrop(224)`, igual ao que o serviço fará em produção.

In [ ]:
# Transformações
MEDIA = [0.485, 0.456, 0.406]
DESVIO = [0.229, 0.224, 0.225]

transf_treino = transforms.Compose(
    [
        transforms.RandomResizedCrop(CONFIG["tamanho_entrada"], scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
        transforms.RandomGrayscale(p=0.05),
        transforms.ToTensor(),
        transforms.Normalize(MEDIA, DESVIO),
    ]
)
transf_avaliacao = transforms.Compose(
    [
        transforms.Resize(CONFIG["redimensionar"]),
        transforms.CenterCrop(CONFIG["tamanho_entrada"]),
        transforms.ToTensor(),
        transforms.Normalize(MEDIA, DESVIO),
    ]
)


class ImagensFolhas(Dataset):
    def __init__(self, amostras: list[tuple[str, object, int]], transformacao, plantdoc=None):
        self.amostras = amostras
        self.transformacao = transformacao
        self.plantdoc = plantdoc

    def __len__(self) -> int:
        return len(self.amostras)

    def imagem(self, i: int) -> Image.Image:
        origem, referencia, _ = self.amostras[i]
        if origem == "pv":
            return Image.open(arquivo_local(referencia)).convert("RGB")
        return self.plantdoc[int(referencia)]["image"].convert("RGB")

    def __getitem__(self, i: int):
        return self.transformacao(self.imagem(i)), self.amostras[i][2]


def amostras_pv(df: pd.DataFrame) -> list[tuple[str, str, int]]:
    return [("pv", c, int(a)) for c, a in zip(df["caminho"], df["alvo"], strict=True)]


def desnormalizar(tensor: torch.Tensor) -> np.ndarray:
    imagem = tensor.cpu().numpy().transpose(1, 2, 0) * np.array(DESVIO) + np.array(MEDIA)
    return np.clip(imagem, 0, 1)


def carregador(conjunto: Dataset, embaralhar=False, amostrador=None) -> DataLoader:
    gerador = torch.Generator().manual_seed(SEED)
    return DataLoader(
        conjunto,
        batch_size=CONFIG["lote"],
        shuffle=embaralhar and amostrador is None,
        sampler=amostrador,
        num_workers=NUM_WORKERS,
        pin_memory=DISPOSITIVO.type == "cuda",
        generator=gerador,
        persistent_workers=NUM_WORKERS > 0,
    )


ds_treino_pv = ImagensFolhas(amostras_pv(treino_pv), transf_treino)
ds_validacao_pv = ImagensFolhas(amostras_pv(validacao_pv), transf_avaliacao)
ds_teste_pv = ImagensFolhas(amostras_pv(teste_pv), transf_avaliacao)

In [ ]:
# Grade de exemplos aumentados
exemplos = treino_pv.groupby("cultura").head(1).index.tolist()[:3] + [treino_pv.index[-1]]
fig, eixos = plt.subplots(len(exemplos), 5, figsize=(13, 2.8 * len(exemplos)))
for linha, i in enumerate(exemplos):
    original = ds_treino_pv.imagem(i)
    eixos[linha, 0].imshow(original)
    eixos[linha, 0].set_title(f"original\n{treino_pv.loc[i, 'rotulo']}", fontsize=8)
    for coluna in range(1, 5):
        eixos[linha, coluna].imshow(desnormalizar(transf_treino(original)))
        eixos[linha, coluna].set_title(f"aumento {coluna}", fontsize=8)
for eixo in eixos.ravel():
    eixo.axis("off")
salvar_figura(fig, "exemplos_aumento.png")

## 3. Experimento 1: MobileNetV3-Large x EfficientNet-B0

**Transfer learning:** em vez de treinar do zero (precisaria de milhões de imagens), partimos de redes já treinadas no ImageNet, que sabem reconhecer bordas, texturas e formas. Trocamos só a última camada, que passa a ter 17 saídas, e ajustamos a rede às folhas.

As duas candidatas são leves, pensadas para rodar em CPU, o que importa porque o serviço de IA roda sem GPU:

| Modelo | Pesos | Parâmetros |
| --- | --- | --- |
| MobileNetV3-Large | `IMAGENET1K_V2` | 5,5 milhões |
| EfficientNet-B0 | `IMAGENET1K_V1` | 5,3 milhões |

O treino tem duas fases:

1. **Backbone congelado (2 épocas, LR 1e-3):** só a cabeça de classificação aprende. Os pesos do ImageNet ficam intactos enquanto a camada nova, ainda aleatória, se ajusta. As camadas de BatchNorm congeladas ficam em modo de avaliação para não alterar suas estatísticas.
2. **Rede inteira (até 6 épocas, LR 1e-4 com decaimento cosseno):** todas as camadas se ajustam devagar, sem destruir o que o modelo já sabia.

Outras escolhas:

- **AdamW** com *weight decay* 1e-4: Adam com regularização dos pesos aplicada do jeito correto.
- **Label smoothing 0.1:** o alvo deixa de ser "100% esta classe" e passa a ser 90% para a classe certa e o resto dividido entre as outras. O modelo fica menos confiante demais, o que ajuda a calibração.
- **Perda ponderada:** cada classe pesa o inverso da sua frequência no treino; errar uma classe rara custa mais.
- **AMP (precisão mista):** na GPU, as contas usam float16 onde é seguro. O treino fica cerca de duas vezes mais rápido e gasta menos memória.
- **Early stopping pelo F1 macro na validação:** guardamos o melhor modelo e paramos se o F1 não melhorar por 2 épocas seguidas.

In [ ]:
# Modelos
ARQUITETURAS = {
    "mobilenet_v3_large": (
        models.mobilenet_v3_large,
        models.MobileNet_V3_Large_Weights.IMAGENET1K_V2,
    ),
    "efficientnet_b0": (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1),
}


def criar_modelo(arquitetura: str, pretreinado: bool = True) -> nn.Module:
    construtor, pesos = ARQUITETURAS[arquitetura]
    modelo = construtor(weights=pesos if pretreinado else None)
    ultima = modelo.classifier[-1]
    modelo.classifier[-1] = nn.Linear(ultima.in_features, NUM_CLASSES)
    return modelo


def pesos_por_classe(alvos) -> torch.Tensor:
    contagem = np.bincount(np.asarray(alvos), minlength=NUM_CLASSES).astype(float)
    pesos = np.where(contagem > 0, contagem.sum() / (NUM_CLASSES * np.maximum(contagem, 1)), 0.0)
    return torch.tensor(pesos, dtype=torch.float32)


def preparar_fase(modelo: nn.Module, fase: str, epocas: int):
    congelar = fase == "congelado"
    for parametro in modelo.features.parameters():
        parametro.requires_grad = not congelar
    treinaveis = [p for p in modelo.parameters() if p.requires_grad]
    lr = CONFIG["lr_cabeca"] if congelar else CONFIG["lr_ajuste"]
    otimizador = torch.optim.AdamW(treinaveis, lr=lr, weight_decay=CONFIG["weight_decay"])
    agendador = torch.optim.lr_scheduler.CosineAnnealingLR(otimizador, T_max=max(epocas, 1))
    return otimizador, agendador


def modo_treino(modelo: nn.Module, fase: str) -> None:
    modelo.train()
    if fase == "congelado":
        modelo.features.eval()


@torch.inference_mode()
def prever(modelo: nn.Module, carregador_dados: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    modelo.eval()
    logits, alvos = [], []
    for imagens, alvo in carregador_dados:
        with torch.autocast(device_type=DISPOSITIVO.type, dtype=torch.float16, enabled=USAR_AMP):
            saida = modelo(imagens.to(DISPOSITIVO, non_blocking=True))
        logits.append(saida.float().cpu())
        alvos.append(alvo)
    return torch.cat(logits).numpy(), torch.cat(alvos).numpy()


def f1_macro(alvos, preditos) -> float:
    return float(
        f1_score(alvos, preditos, labels=np.unique(alvos), average="macro", zero_division=0)
    )

In [ ]:
# Laço de treino com checkpoint a cada época
def treinar(
    nome: str,
    arquitetura: str,
    dl_treino: DataLoader,
    dl_validacao: DataLoader,
    pesos: torch.Tensor,
):
    arquivo_estado = CHECKPOINTS / f"{nome}_estado.pt"
    arquivo_melhor = CHECKPOINTS / f"{nome}_melhor.pt"
    plano = ["congelado"] * CONFIG["epocas_congelado"] + ["descongelado"] * CONFIG[
        "epocas_descongelado"
    ]

    modelo = criar_modelo(arquitetura).to(DISPOSITIVO)
    criterio = nn.CrossEntropyLoss(
        weight=pesos.to(DISPOSITIVO), label_smoothing=CONFIG["label_smoothing"]
    )
    escalador = torch.amp.GradScaler("cuda", enabled=USAR_AMP)

    estado = (
        torch.load(arquivo_estado, map_location="cpu", weights_only=False)
        if arquivo_estado.exists()
        else None
    )
    if estado and estado["concluido"]:
        modelo.load_state_dict(torch.load(arquivo_melhor, map_location=DISPOSITIVO))
        print(f"{nome}: treino já concluído, carregado do checkpoint.")
        return modelo, estado["historico"]

    inicio, historico, melhor_f1, sem_melhora = 0, [], -1.0, 0
    if estado:
        modelo.load_state_dict(estado["modelo"])
        inicio, historico = estado["proxima_epoca"], estado["historico"]
        melhor_f1, sem_melhora = estado["melhor_f1"], estado["sem_melhora"]
        print(f"{nome}: retomando da época {inicio + 1}.")

    fase_atual = None
    for epoca in range(inicio, len(plano)):
        fase = plano[epoca]
        if fase != fase_atual:
            otimizador, agendador = preparar_fase(modelo, fase, plano.count(fase))
            if estado and epoca == inicio and estado["fase"] == fase:
                otimizador.load_state_dict(estado["otimizador"])
                agendador.load_state_dict(estado["agendador"])
                escalador.load_state_dict(estado["escalador"])
            elif fase == "descongelado":
                sem_melhora = 0
            fase_atual = fase

        comeco = time.time()
        modo_treino(modelo, fase)
        soma_perda, total = 0.0, 0
        for imagens, alvos in dl_treino:
            imagens = imagens.to(DISPOSITIVO, non_blocking=True)
            alvos = alvos.to(DISPOSITIVO, non_blocking=True)
            otimizador.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=DISPOSITIVO.type, dtype=torch.float16, enabled=USAR_AMP
            ):
                perda = criterio(modelo(imagens), alvos)
            escalador.scale(perda).backward()
            escalador.step(otimizador)
            escalador.update()
            soma_perda += perda.item() * len(alvos)
            total += len(alvos)
        agendador.step()

        logits_val, alvos_val = prever(modelo, dl_validacao)
        perda_val = F.cross_entropy(
            torch.from_numpy(logits_val), torch.from_numpy(alvos_val)
        ).item()
        f1_val = f1_macro(alvos_val, logits_val.argmax(1))
        historico.append(
            {
                "epoca": epoca + 1,
                "fase": fase,
                "perda_treino": soma_perda / total,
                "perda_validacao": perda_val,
                "f1_validacao": f1_val,
                "segundos": time.time() - comeco,
            }
        )
        print(
            f"{nome} época {epoca + 1}/{len(plano)} ({fase}): perda {soma_perda / total:.4f}, "
            f"perda val {perda_val:.4f}, F1 macro val {f1_val:.4f}, {time.time() - comeco:.0f} s"
        )

        if f1_val > melhor_f1:
            melhor_f1, sem_melhora = f1_val, 0
            torch.save(modelo.state_dict(), arquivo_melhor)
        else:
            sem_melhora += 1

        parar = fase == "descongelado" and sem_melhora >= CONFIG["paciencia"]
        torch.save(
            {
                "modelo": modelo.state_dict(),
                "otimizador": otimizador.state_dict(),
                "agendador": agendador.state_dict(),
                "escalador": escalador.state_dict(),
                "fase": fase,
                "proxima_epoca": epoca + 1,
                "historico": historico,
                "melhor_f1": melhor_f1,
                "sem_melhora": sem_melhora,
                "concluido": parar or epoca + 1 == len(plano),
            },
            arquivo_estado,
        )
        if parar:
            print(f"{nome}: early stopping, F1 sem melhora por {sem_melhora} épocas.")
            break

    estado_final = torch.load(arquivo_estado, map_location="cpu", weights_only=False)
    estado_final["concluido"] = True
    torch.save(estado_final, arquivo_estado)
    modelo.load_state_dict(torch.load(arquivo_melhor, map_location=DISPOSITIVO))
    return modelo, historico

In [ ]:
# Treino das duas arquiteturas
dl_treino_pv = carregador(ds_treino_pv, embaralhar=True)
dl_validacao_pv = carregador(ds_validacao_pv)
dl_teste_pv = carregador(ds_teste_pv)
pesos_pv = pesos_por_classe(treino_pv["alvo"])

MODELOS_EXP1, HISTORICOS = {}, {}
for arquitetura in ARQUITETURAS:
    fixar_seed(SEED)
    MODELOS_EXP1[arquitetura], HISTORICOS[arquitetura] = treinar(
        f"exp1_{arquitetura}", arquitetura, dl_treino_pv, dl_validacao_pv, pesos_pv
    )

In [ ]:
# Curvas de perda e F1
fig, eixos = plt.subplots(1, 2, figsize=(13, 4))
for arquitetura, historico in HISTORICOS.items():
    h = pd.DataFrame(historico)
    eixos[0].plot(h["epoca"], h["perda_treino"], marker="o", label=f"{arquitetura} treino")
    eixos[0].plot(
        h["epoca"],
        h["perda_validacao"],
        marker="o",
        linestyle="--",
        label=f"{arquitetura} validação",
    )
    eixos[1].plot(h["epoca"], h["f1_validacao"], marker="o", label=arquitetura)
for eixo in eixos:
    eixo.axvline(
        CONFIG["epocas_congelado"] + 0.5, color="gray", linestyle=":", label="descongela o backbone"
    )
    eixo.set_xlabel("época")
    eixo.legend(fontsize=8)
eixos[0].set_title("Perda (treino com label smoothing e pesos; validação sem)")
eixos[1].set_title("F1 macro na validação")
salvar_figura(fig, "curvas_experimento1.png")

## 4. Comparação no teste do PlantVillage

O conjunto de teste não foi usado em nenhuma decisão até aqui. Para cada modelo medimos:

- **acurácia:** fração de acertos;
- **precisão, recall e F1 por classe:** precisão responde "quando o modelo diz requeima, quantas vezes é requeima?"; recall responde "das folhas com requeima, quantas ele encontrou?"; F1 é a média harmônica das duas;
- **F1 macro:** média simples do F1 das 17 classes, sem favorecer as classes grandes;
- **matriz de confusão normalizada:** cada linha mostra para onde vão as imagens de uma classe;
- **tamanho do arquivo e latência em CPU:** o serviço roda sem GPU, então isso pesa na escolha.

**Critério de escolha:** maior F1 macro no teste. Se a diferença for menor que 0,5 ponto percentual, fica o modelo com menor latência em CPU.

In [ ]:
# Métricas e matriz de confusão
def metricas(alvos, preditos) -> dict:
    presentes = np.unique(alvos)
    relatorio = classification_report(
        alvos,
        preditos,
        labels=presentes,
        target_names=[ROTULOS[i] for i in presentes],
        output_dict=True,
        zero_division=0,
    )
    return {
        "acuracia": float(accuracy_score(alvos, preditos)),
        "f1Macro": f1_macro(alvos, preditos),
        "precisaoMacro": float(relatorio["macro avg"]["precision"]),
        "recallMacro": float(relatorio["macro avg"]["recall"]),
        "porClasse": {k: v for k, v in relatorio.items() if k in ROTULOS},
    }


def imprimir_relatorio(alvos, preditos) -> None:
    presentes = np.unique(alvos)
    print(
        classification_report(
            alvos,
            preditos,
            labels=presentes,
            target_names=[ROTULOS[i] for i in presentes],
            digits=3,
            zero_division=0,
        )
    )


def plotar_matriz(alvos, preditos, titulo: str, arquivo: str) -> np.ndarray:
    matriz = confusion_matrix(alvos, preditos, labels=range(NUM_CLASSES)).astype(float)
    soma = matriz.sum(axis=1, keepdims=True)
    normalizada = np.divide(matriz, soma, out=np.full_like(matriz, np.nan), where=soma > 0)
    fig, ax = plt.subplots(figsize=(11, 9))
    imagem = ax.imshow(normalizada, cmap="Greens", vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLASSES), ROTULOS, rotation=90, fontsize=8)
    ax.set_yticks(range(NUM_CLASSES), ROTULOS, fontsize=8)
    ax.set_xlabel("classe prevista")
    ax.set_ylabel("classe real")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            if not np.isnan(normalizada[i, j]) and normalizada[i, j] >= 0.01:
                cor = "white" if normalizada[i, j] > 0.6 else "black"
                ax.text(
                    j,
                    i,
                    f"{normalizada[i, j]:.2f}",
                    ha="center",
                    va="center",
                    fontsize=6,
                    color=cor,
                )
    ax.set_title(titulo)
    fig.colorbar(imagem, ax=ax, fraction=0.046)
    salvar_figura(fig, arquivo)
    return matriz


def tamanho_mb(modelo: nn.Module) -> float:
    with tempfile.TemporaryDirectory() as pasta:
        caminho = Path(pasta) / "pesos.pt"
        torch.save(modelo.state_dict(), caminho)
        return caminho.stat().st_size / 2**20


@torch.inference_mode()
def latencia_cpu_ms(modelo: nn.Module, repeticoes: int = 5 if MODO_RAPIDO else 50) -> float:
    copia = copy.deepcopy(modelo).cpu().eval()
    entrada = torch.randn(1, 3, CONFIG["tamanho_entrada"], CONFIG["tamanho_entrada"])
    for _ in range(3):
        copia(entrada)
    inicio = time.perf_counter()
    for _ in range(repeticoes):
        copia(entrada)
    return (time.perf_counter() - inicio) / repeticoes * 1000

In [ ]:
# Avaliação no teste do PlantVillage
LOGITS_TESTE_PV, COMPARACAO = {}, []
for arquitetura, modelo in MODELOS_EXP1.items():
    logits, alvos_teste_pv = prever(modelo, dl_teste_pv)
    LOGITS_TESTE_PV[arquitetura] = logits
    preditos = logits.argmax(1)
    print(f"=== {arquitetura} ===")
    imprimir_relatorio(alvos_teste_pv, preditos)
    matriz = plotar_matriz(
        alvos_teste_pv,
        preditos,
        f"{arquitetura}: teste PlantVillage (linhas normalizadas)",
        f"matriz_confusao_exp1_{arquitetura}.png",
    )
    resultado = {
        "arquitetura": arquitetura,
        **metricas(alvos_teste_pv, preditos),
        "tamanhoMb": tamanho_mb(modelo),
        "latenciaCpuMs": latencia_cpu_ms(modelo),
        "historico": HISTORICOS[arquitetura],
        "matrizConfusao": matriz,
    }
    salvar_json(f"experimento1_{arquitetura}.json", resultado)
    COMPARACAO.append(resultado)

tabela = pd.DataFrame(COMPARACAO).set_index("arquitetura")[
    ["acuracia", "f1Macro", "precisaoMacro", "recallMacro", "tamanhoMb", "latenciaCpuMs"]
]
display(
    tabela.style.format(
        {
            "acuracia": "{:.4f}",
            "f1Macro": "{:.4f}",
            "precisaoMacro": "{:.4f}",
            "recallMacro": "{:.4f}",
            "tamanhoMb": "{:.1f}",
            "latenciaCpuMs": "{:.1f}",
        }
    )
)

In [ ]:
# Escolha da arquitetura
ordenada = tabela.sort_values("f1Macro", ascending=False)
primeira, segunda = ordenada.index[0], ordenada.index[1]
diferenca = ordenada.loc[primeira, "f1Macro"] - ordenada.loc[segunda, "f1Macro"]
if diferenca < 0.005:
    ESCOLHIDA = tabela["latenciaCpuMs"].idxmin()
    motivo = (
        f"A diferença de F1 macro é de só {diferenca * 100:.2f} p.p., dentro do critério de "
        "empate, "
        f"então fica o modelo mais rápido em CPU."
    )
else:
    ESCOLHIDA = primeira
    motivo = f"O F1 macro é {diferenca * 100:.2f} p.p. maior que o do outro modelo."
outra = segunda if primeira == ESCOLHIDA else primeira
linha, alternativa = tabela.loc[ESCOLHIDA], tabela.loc[outra]
MODELO_EXP1 = MODELOS_EXP1[ESCOLHIDA]
display(
    Markdown(
        f"**Arquitetura escolhida: `{ESCOLHIDA}`.** {motivo} No teste do PlantVillage ela "
        "teve acurácia de "
        f"{linha['acuracia']:.2%} e F1 macro de {linha['f1Macro']:.4f}, contra "
        f"{alternativa['f1Macro']:.4f} da `{outra}`. "
        f"O arquivo tem {linha['tamanhoMb']:.1f} MB e a inferência leva "
        f"{linha['latenciaCpuMs']:.0f} ms por imagem em CPU "
        f"({alternativa['latenciaCpuMs']:.0f} ms na `{outra}`), o que cabe no tempo limite da "
        "triagem."
    )
)
RESULTADOS["experimento1"] = {
    "escolhida": ESCOLHIDA,
    "motivo": motivo,
    "tabela": tabela.reset_index().to_dict(orient="records"),
}

## 5. Calibração da confiança

O serviço só manda alerta quando a confiança passa de `CONFIANCA_MINIMA` (0,6). Para esse limite fazer sentido, "0,8 de confiança" precisa significar "acerta cerca de 80% das vezes". Redes neurais costumam ser **confiantes demais**.

**Temperature scaling:** dividimos as saídas da rede (logits) por um número *T* antes do softmax. Com *T* > 1 as probabilidades ficam menos extremas (modelo confiante demais); com *T* < 1 ficam mais extremas (modelo pouco confiante, efeito comum do label smoothing). *T* é escolhido na **validação**, minimizando a perda logarítmica. O ranking das classes não muda, então a acurácia continua a mesma; muda só o quanto o modelo diz confiar.

**ECE (erro de calibração esperado):** separamos as previsões em 15 faixas de confiança e comparamos, em cada faixa, a confiança média com a acurácia real. O ECE é a média ponderada dessas diferenças; quanto menor, melhor.

**Renormalização na cultura:** em produção o produtor informa a cultura, e o serviço considera só as classes dela, renormalizando as probabilidades. Mostramos as curvas também desse jeito.

A **curva de cobertura x acurácia** mostra, para cada limite de confiança de 0,3 a 0,95, quantos casos passam do limite (cobertura) e quantos deles estão certos (acurácia). Os que não passam vão para o agrônomo sem alerta.

In [ ]:
# Temperature scaling e ECE
def softmax(logits: np.ndarray, temperatura: float = 1.0) -> np.ndarray:
    z = logits / temperatura
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def softmax_na_cultura(logits: np.ndarray, culturas, temperatura: float = 1.0) -> np.ndarray:
    mascara = CULTURAS[None, :] == np.asarray(culturas)[:, None]
    z = np.where(mascara, logits / temperatura, -np.inf)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def ajustar_temperatura(logits: np.ndarray, alvos: np.ndarray) -> float:
    saidas, corretos = torch.from_numpy(logits).float(), torch.from_numpy(alvos).long()
    log_t = torch.zeros(1, requires_grad=True)
    otimizador = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)

    def fechamento():
        otimizador.zero_grad()
        perda = F.cross_entropy(saidas / log_t.exp(), corretos)
        perda.backward()
        return perda

    otimizador.step(fechamento)
    return float(log_t.exp().item())


def ece(probabilidades: np.ndarray, alvos: np.ndarray, faixas: int = 15) -> float:
    confianca, preditos = probabilidades.max(1), probabilidades.argmax(1)
    acertos = preditos == alvos
    limites = np.linspace(0, 1, faixas + 1)
    total = 0.0
    for inferior, superior in zip(limites[:-1], limites[1:], strict=True):
        na_faixa = (confianca > inferior) & (confianca <= superior)
        if na_faixa.any():
            total += na_faixa.mean() * abs(acertos[na_faixa].mean() - confianca[na_faixa].mean())
    return float(total)


def diagrama_confiabilidade(ax, probabilidades, alvos, titulo, faixas=15):
    confianca, acertos = probabilidades.max(1), probabilidades.argmax(1) == alvos
    limites = np.linspace(0, 1, faixas + 1)
    centros, acuracias = [], []
    for inferior, superior in zip(limites[:-1], limites[1:], strict=True):
        na_faixa = (confianca > inferior) & (confianca <= superior)
        if na_faixa.any():
            centros.append((inferior + superior) / 2)
            acuracias.append(acertos[na_faixa].mean())
    ax.bar(
        centros,
        acuracias,
        width=1 / faixas,
        edgecolor="black",
        color="#81c784",
        label="acurácia na faixa",
    )
    ax.plot([0, 1], [0, 1], "k--", label="calibração perfeita")
    ax.set_xlabel("confiança")
    ax.set_ylabel("acurácia")
    ax.set_title(f"{titulo}\nECE = {ece(probabilidades, alvos, faixas):.4f}")
    ax.legend(fontsize=8)


logits_val_exp1, alvos_val_pv = prever(MODELO_EXP1, dl_validacao_pv)
TEMPERATURA_EXP1 = ajustar_temperatura(logits_val_exp1, alvos_val_pv)
logits_teste_exp1 = LOGITS_TESTE_PV[ESCOLHIDA]
culturas_teste_pv = teste_pv["cultura"].to_numpy()

prob_antes = softmax(logits_teste_exp1)
prob_depois = softmax(logits_teste_exp1, TEMPERATURA_EXP1)
ece_antes, ece_depois = ece(prob_antes, alvos_teste_pv), ece(prob_depois, alvos_teste_pv)
print(f"Temperatura ajustada na validação: T = {TEMPERATURA_EXP1:.3f}")
print(f"ECE no teste: {ece_antes:.4f} antes e {ece_depois:.4f} depois")

fig, eixos = plt.subplots(1, 2, figsize=(11, 4.5))
diagrama_confiabilidade(eixos[0], prob_antes, alvos_teste_pv, "Antes (T = 1)")
diagrama_confiabilidade(
    eixos[1], prob_depois, alvos_teste_pv, f"Depois (T = {TEMPERATURA_EXP1:.2f})"
)
salvar_figura(fig, "calibracao_confiabilidade.png")

In [ ]:
# Cobertura x acurácia
LIMITES = np.round(np.arange(0.30, 0.951, 0.05), 2)


def cobertura_acuracia(probabilidades: np.ndarray, alvos: np.ndarray) -> pd.DataFrame:
    confianca, acertos = probabilidades.max(1), probabilidades.argmax(1) == alvos
    linhas = []
    for limite in LIMITES:
        aceitos = confianca >= limite
        linhas.append(
            {
                "limite": limite,
                "cobertura": aceitos.mean(),
                "acuracia": acertos[aceitos].mean() if aceitos.any() else np.nan,
            }
        )
    return pd.DataFrame(linhas)


curva_17 = cobertura_acuracia(prob_depois, alvos_teste_pv)
curva_cultura = cobertura_acuracia(
    softmax_na_cultura(logits_teste_exp1, culturas_teste_pv, TEMPERATURA_EXP1), alvos_teste_pv
)

fig, ax = plt.subplots(figsize=(8, 4.5))
for curva, nome, estilo in (
    (curva_17, "17 classes", "-"),
    (curva_cultura, "renormalizado na cultura", "--"),
):
    ax.plot(
        curva["limite"],
        curva["cobertura"],
        estilo,
        marker="o",
        color="#1565c0",
        label=f"cobertura ({nome})",
    )
    ax.plot(
        curva["limite"],
        curva["acuracia"],
        estilo,
        marker="s",
        color="#2e7d32",
        label=f"acurácia ({nome})",
    )
ax.axvline(0.6, color="#c62828", linestyle=":", label="CONFIANCA_MINIMA = 0,6")
ax.set_xlabel("limite de confiança")
ax.set_ylim(0, 1.02)
ax.set_title("Teste PlantVillage, confiança calibrada")
ax.legend(fontsize=8)
salvar_figura(fig, "cobertura_acuracia.png")

em_06 = curva_cultura.set_index("limite").loc[0.6]
display(
    Markdown(
        f"Com o limite de **0,6** e a renormalização na cultura, {em_06['cobertura']:.1%} das "
        "fotos do teste recebem alerta, "
        f"e a acurácia entre elas é de {em_06['acuracia']:.1%}. As outras "
        f"{1 - em_06['cobertura']:.1%} vão para o agrônomo sem alerta."
    )
)
RESULTADOS["calibracaoExp1"] = {
    "temperatura": TEMPERATURA_EXP1,
    "eceAntes": ece_antes,
    "eceDepois": ece_depois,
    "curva17": curva_17.to_dict(orient="records"),
    "curvaCultura": curva_cultura.to_dict(orient="records"),
}
salvar_json(
    "calibracao_experimento1.json", {"arquitetura": ESCOLHIDA, **RESULTADOS["calibracaoExp1"]}
)

## 6. Experimento 2: fotos de campo (PlantDoc), sem ajuste

O PlantVillage foi fotografado em laboratório, com fundo liso e luz controlada. O produtor vai fotografar no campo, com solo, outras folhas, sombra e a mão no quadro. O **PlantDoc** (Singh et al., 2020) reúne fotos de campo tiradas da internet, com classes parecidas.

Aqui avaliamos o modelo escolhido **sem nenhum treino** com essas fotos, para medir a queda de desempenho fora do laboratório. As classes do PlantDoc são mapeadas para os rótulos do Valence:

| PlantDoc | Rótulo Valence |
| --- | --- |
| Corn Gray leaf spot | `milho_cercosporiose` |
| Corn leaf blight | `milho_helmintosporiose` |
| Corn rust leaf | `milho_ferrugem_comum` |
| Potato leaf early blight | `batata_pinta_preta` |
| Potato leaf late blight | `batata_requeima` |
| Tomato Early blight leaf | `tomate_pinta_preta` |
| Tomato Septoria leaf spot | `tomate_septoriose` |
| Tomato leaf | `tomate_saudavel` |
| Tomato leaf bacterial spot | `tomate_mancha_bacteriana` |
| Tomato leaf late blight | `tomate_requeima` |
| Tomato leaf mosaic virus | `tomate_virus_mosaico` |
| Tomato leaf yellow virus | `tomate_virus_enrolamento_amarelo` |
| Tomato mold leaf | `tomate_mofo_das_folhas` |

`Tomato two spotted spider mites leaf` fica de fora: são só 2 imagens. Milho saudável, batata saudável e mancha-alvo do tomate não existem no PlantDoc.

Reportamos dois números: a previsão entre as **17 classes** e a previsão **renormalizada na cultura informada**, que é como o serviço funciona.

As fotos das outras culturas do PlantDoc (maçã, uva, soja e outras) são exportadas para avaliar o portão, que deve recusá-las como cultura fora do escopo.

In [ ]:
# Carga do PlantDoc
REPO_PD = "Project-AgML/plant_doc_classification"
plantdoc = datasets.load_dataset(REPO_PD)
print(plantdoc)

if "test" in plantdoc:
    tamanho_treino_pd = len(plantdoc["train"])
    PLANTDOC = datasets.concatenate_datasets([plantdoc["train"], plantdoc["test"]])
    SPLIT_ORIGINAL = np.array(["train"] * tamanho_treino_pd + ["test"] * len(plantdoc["test"]))
else:
    PLANTDOC = plantdoc["train"]
    SPLIT_ORIGINAL = np.array(["train"] * len(PLANTDOC))
TEM_TESTE_OFICIAL_PD = "test" in plantdoc

NOMES_PD = PLANTDOC.features["label"].names
MAPA_PLANTDOC = {
    "Corn Gray leaf spot": "milho_cercosporiose",
    "Corn leaf blight": "milho_helmintosporiose",
    "Corn rust leaf": "milho_ferrugem_comum",
    "Potato leaf early blight": "batata_pinta_preta",
    "Potato leaf late blight": "batata_requeima",
    "Tomato Early blight leaf": "tomate_pinta_preta",
    "Tomato Septoria leaf spot": "tomate_septoriose",
    "Tomato leaf": "tomate_saudavel",
    "Tomato leaf bacterial spot": "tomate_mancha_bacteriana",
    "Tomato leaf late blight": "tomate_requeima",
    "Tomato leaf mosaic virus": "tomate_virus_mosaico",
    "Tomato leaf yellow virus": "tomate_virus_enrolamento_amarelo",
    "Tomato mold leaf": "tomate_mofo_das_folhas",
}
IGNORADAS_PD = {"Tomato two spotted spider mites leaf"}
das_culturas = {n for n in NOMES_PD if n.split()[0] in ("Corn", "Potato", "Tomato")}
assert set(MAPA_PLANTDOC) <= set(NOMES_PD)
assert das_culturas == set(MAPA_PLANTDOC) | IGNORADAS_PD, das_culturas ^ (
    set(MAPA_PLANTDOC) | IGNORADAS_PD
)

classes_pd = np.array([NOMES_PD[i] for i in PLANTDOC["label"]])
tabela_pd = pd.DataFrame(
    {"indice": np.arange(len(PLANTDOC)), "classe": classes_pd, "split": SPLIT_ORIGINAL}
)
tabela_pd["rotulo"] = tabela_pd["classe"].map(MAPA_PLANTDOC)
mapeadas_pd = tabela_pd.dropna(subset=["rotulo"]).copy()
mapeadas_pd["alvo"] = mapeadas_pd["rotulo"].map(INDICE_DO_ROTULO).astype(int)
mapeadas_pd["cultura"] = CULTURAS[mapeadas_pd["alvo"]]
outras_pd = tabela_pd[~tabela_pd["classe"].isin(das_culturas)]

if MODO_RAPIDO:
    mapeadas_pd = (
        mapeadas_pd.sample(frac=1.0, random_state=SEED)
        .groupby("alvo")
        .head(CONFIG["rapido_plantdoc_por_classe"])
    )
    outras_pd = outras_pd.sample(
        min(CONFIG["rapido_plantdoc_outras"], len(outras_pd)), random_state=SEED
    )
mapeadas_pd = mapeadas_pd.sort_values("indice").reset_index(drop=True)

display(mapeadas_pd.groupby(["classe", "rotulo"]).size().rename("imagens").to_frame())
print(
    f"Imagens mapeadas: {len(mapeadas_pd)}; de outras culturas: {len(outras_pd)}; ignoradas: "
    f"{int(tabela_pd['classe'].isin(IGNORADAS_PD).sum())}"
)

salvar_json(
    "plantdoc_outras_culturas.json",
    {
        "dataset": REPO_PD,
        "uso": "Fotos de campo de culturas fora do escopo, para avaliar se o portão as recusa "
        "(CULTURA_FORA_DO_ESCOPO).",
        "modoRapido": MODO_RAPIDO,
        "total": len(outras_pd),
        "porClasse": outras_pd["classe"].value_counts().to_dict(),
        "imagens": outras_pd[["indice", "split", "classe"]].to_dict(orient="records"),
    },
)

In [ ]:
# Avaliação sem ajuste
def amostras_pd(df: pd.DataFrame) -> list[tuple[str, int, int]]:
    return [("pd", int(i), int(a)) for i, a in zip(df["indice"], df["alvo"], strict=True)]


def avaliar_campo(alvos, logits, culturas, temperatura: float) -> dict:
    preditos_17 = logits.argmax(1)
    preditos_cultura = softmax_na_cultura(logits, culturas, temperatura).argmax(1)
    return {
        "17classes": metricas(alvos, preditos_17),
        "naCultura": metricas(alvos, preditos_cultura),
    }


dl_plantdoc = carregador(ImagensFolhas(amostras_pd(mapeadas_pd), transf_avaliacao, PLANTDOC))
logits_pd_exp1, alvos_pd = prever(MODELO_EXP1, dl_plantdoc)
culturas_pd = mapeadas_pd["cultura"].to_numpy()
exp2 = avaliar_campo(alvos_pd, logits_pd_exp1, culturas_pd, TEMPERATURA_EXP1)

print("Renormalizado na cultura (como em produção):")
imprimir_relatorio(alvos_pd, softmax_na_cultura(logits_pd_exp1, culturas_pd).argmax(1))
matriz_exp2 = plotar_matriz(
    alvos_pd,
    softmax_na_cultura(logits_pd_exp1, culturas_pd).argmax(1),
    f"{ESCOLHIDA} sem ajuste: PlantDoc, renormalizado na cultura",
    "matriz_confusao_exp2_plantdoc.png",
)

f1_pv = tabela.loc[ESCOLHIDA, "f1Macro"]
display(
    Markdown(
        f"**Queda fora do laboratório:** o F1 macro cai de {f1_pv:.4f} no teste do "
        "PlantVillage para "
        f"{exp2['17classes']['f1Macro']:.4f} no PlantDoc entre as 17 classes e "
        f"{exp2['naCultura']['f1Macro']:.4f} "
        "quando a cultura é informada. A acurácia renormalizada na cultura é "
        f"{exp2['naCultura']['acuracia']:.1%}."
    )
)
RESULTADOS["experimento2"] = {"arquitetura": ESCOLHIDA, "imagens": len(mapeadas_pd), **exp2}
salvar_json("experimento2_plantdoc.json", RESULTADOS["experimento2"])

## 7. Experimento 3: ajuste fino com fotos de campo

Agora o modelo escolhido é treinado de novo, a partir do ImageNet, com o treino do PlantVillage **mais** uma parte do PlantDoc.

- **Divisão do PlantDoc:** este dataset publica só o split de treino. Separamos 30% para teste, estratificado por classe e com a mesma seed, e 15% do restante para validação.
- **Amostragem por domínio:** o PlantDoc tem poucas centenas de imagens perto de milhares do PlantVillage. Um amostrador ponderado faz cerca de 30% de cada época vir do PlantDoc, para as fotos de campo não se perderem no meio das de laboratório.
- **Mesmos hiperparâmetros** do Experimento 1, para a comparação ser justa.

A decisão de produção usa o **mesmo teste do PlantDoc** nos dois modelos. O modelo ajustado vai para produção se melhorar o F1 macro no campo sem perder mais de 2 pontos percentuais no teste do PlantVillage.

**Atenção:** o PlantDoc não identifica a folha nem a planta de origem, então fotos muito parecidas podem cair no treino e no teste. O resultado no PlantDoc pode ser um pouco otimista.

In [ ]:
# Divisão do PlantDoc
def dividir_estratificado(df: pd.DataFrame, fracao: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    try:
        a, b = train_test_split(df, test_size=fracao, stratify=df["alvo"], random_state=SEED)
    except ValueError:
        a, b = train_test_split(df, test_size=fracao, random_state=SEED)
    return a.reset_index(drop=True), b.reset_index(drop=True)


if TEM_TESTE_OFICIAL_PD:
    treino_pd_total = mapeadas_pd[mapeadas_pd["split"] == "train"].reset_index(drop=True)
    teste_pd = mapeadas_pd[mapeadas_pd["split"] == "test"].reset_index(drop=True)
    print("Usando o split de teste oficial do PlantDoc.")
else:
    treino_pd_total, teste_pd = dividir_estratificado(mapeadas_pd, CONFIG["fracao_teste_plantdoc"])
    print(
        "Sem split de teste oficial: "
        f"{1 - CONFIG['fracao_teste_plantdoc']:.0%}/{CONFIG['fracao_teste_plantdoc']:.0%} "
        f"estratificado, seed {SEED}."
    )
treino_pd, validacao_pd = dividir_estratificado(treino_pd_total, CONFIG["fracao_validacao"])
print(f"PlantDoc: treino {len(treino_pd)}, validação {len(validacao_pd)}, teste {len(teste_pd)}")

amostras_treino_exp3 = amostras_pv(treino_pv) + amostras_pd(treino_pd)
fracao_pd = CONFIG["fracao_plantdoc_no_lote"]
pesos_amostras = [
    (1 - fracao_pd) / len(treino_pv) if origem == "pv" else fracao_pd / len(treino_pd)
    for origem, _, _ in amostras_treino_exp3
]
amostrador = WeightedRandomSampler(
    pesos_amostras,
    num_samples=len(amostras_treino_exp3),
    replacement=True,
    generator=torch.Generator().manual_seed(SEED),
)
dl_treino_exp3 = carregador(
    ImagensFolhas(amostras_treino_exp3, transf_treino, PLANTDOC), amostrador=amostrador
)
dl_validacao_exp3 = carregador(
    ImagensFolhas(amostras_pv(validacao_pv) + amostras_pd(validacao_pd), transf_avaliacao, PLANTDOC)
)
dl_teste_pd = carregador(ImagensFolhas(amostras_pd(teste_pd), transf_avaliacao, PLANTDOC))
pesos_exp3 = pesos_por_classe([a for _, _, a in amostras_treino_exp3])

In [ ]:
# Treino com PlantVillage e PlantDoc
fixar_seed(SEED)
MODELO_EXP3, HISTORICO_EXP3 = treinar(
    f"exp3_{ESCOLHIDA}", ESCOLHIDA, dl_treino_exp3, dl_validacao_exp3, pesos_exp3
)

h = pd.DataFrame(HISTORICO_EXP3)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(
    h["epoca"], h["f1_validacao"], marker="o", label="F1 macro validação (PlantVillage + PlantDoc)"
)
ax.plot(h["epoca"], h["perda_validacao"], marker="s", linestyle="--", label="perda validação")
ax.axvline(CONFIG["epocas_congelado"] + 0.5, color="gray", linestyle=":")
ax.set_xlabel("época")
ax.legend(fontsize=8)
ax.set_title(f"Experimento 3: {ESCOLHIDA}")
salvar_figura(fig, "curvas_experimento3.png")

In [ ]:
# Comparação entre o Experimento 2 e o Experimento 3
posicao_no_exp2 = {indice: i for i, indice in enumerate(mapeadas_pd["indice"])}
linhas_teste_pd = [posicao_no_exp2[i] for i in teste_pd["indice"]]
alvos_teste_pd = teste_pd["alvo"].to_numpy()
culturas_teste_pd = teste_pd["cultura"].to_numpy()

logits_pv_exp3, _ = prever(MODELO_EXP3, dl_teste_pv)
logits_pd_exp3, _ = prever(MODELO_EXP3, dl_teste_pd)
logits_val_exp3, alvos_val_exp3 = prever(MODELO_EXP3, dl_validacao_exp3)
TEMPERATURA_EXP3 = ajustar_temperatura(logits_val_exp3, alvos_val_exp3)

comparacao = {
    "exp1": {
        "plantvillage": metricas(alvos_teste_pv, logits_teste_exp1.argmax(1)),
        "plantdoc": avaliar_campo(
            alvos_teste_pd, logits_pd_exp1[linhas_teste_pd], culturas_teste_pd, TEMPERATURA_EXP1
        ),
    },
    "exp3": {
        "plantvillage": metricas(alvos_teste_pv, logits_pv_exp3.argmax(1)),
        "plantdoc": avaliar_campo(
            alvos_teste_pd, logits_pd_exp3, culturas_teste_pd, TEMPERATURA_EXP3
        ),
    },
}
tabela_exp3 = pd.DataFrame(
    {
        nome: {
            "PlantVillage acurácia": r["plantvillage"]["acuracia"],
            "PlantVillage F1 macro": r["plantvillage"]["f1Macro"],
            "PlantDoc acurácia (cultura)": r["plantdoc"]["naCultura"]["acuracia"],
            "PlantDoc F1 macro (cultura)": r["plantdoc"]["naCultura"]["f1Macro"],
            "PlantDoc F1 macro (17 classes)": r["plantdoc"]["17classes"]["f1Macro"],
        }
        for nome, r in comparacao.items()
    }
).T.rename(index={"exp1": "Exp. 1 (só PlantVillage)", "exp3": "Exp. 3 (com PlantDoc)"})
display(tabela_exp3.style.format("{:.4f}"))

print("Experimento 3 no teste do PlantDoc, renormalizado na cultura:")
preditos_pd_exp3 = softmax_na_cultura(logits_pd_exp3, culturas_teste_pd).argmax(1)
imprimir_relatorio(alvos_teste_pd, preditos_pd_exp3)
matriz_exp3_pd = plotar_matriz(
    alvos_teste_pd,
    preditos_pd_exp3,
    f"Exp. 3 {ESCOLHIDA}: teste PlantDoc, renormalizado na cultura",
    "matriz_confusao_exp3_plantdoc.png",
)
matriz_exp3_pv = plotar_matriz(
    alvos_teste_pv,
    logits_pv_exp3.argmax(1),
    f"Exp. 3 {ESCOLHIDA}: teste PlantVillage",
    "matriz_confusao_exp3_plantvillage.png",
)

In [ ]:
# Decisão de produção
ganho_campo = (
    comparacao["exp3"]["plantdoc"]["naCultura"]["f1Macro"]
    - comparacao["exp1"]["plantdoc"]["naCultura"]["f1Macro"]
)
perda_lab = (
    comparacao["exp1"]["plantvillage"]["f1Macro"] - comparacao["exp3"]["plantvillage"]["f1Macro"]
)
if ganho_campo > 0 and perda_lab <= 0.02:
    EXPERIMENTO_PRODUCAO, MODELO_PRODUCAO = "exp3", MODELO_EXP3
    decisao = (
        f"O ajuste fino melhorou o F1 macro no campo em {ganho_campo * 100:.1f} p.p. e mudou "
        "o F1 no "
        f"PlantVillage em {-perda_lab * 100:+.1f} p.p., dentro da tolerância de 2 p.p."
    )
else:
    EXPERIMENTO_PRODUCAO, MODELO_PRODUCAO = "exp1", MODELO_EXP1
    decisao = (
        f"O ajuste fino mudou o F1 macro no campo em {ganho_campo * 100:+.1f} p.p. e o do "
        "PlantVillage em "
        f"{-perda_lab * 100:+.1f} p.p.; não compensou trocar o modelo."
    )
display(
    Markdown(
        f"**Modelo de produção: Experimento {EXPERIMENTO_PRODUCAO[-1]} (`{ESCOLHIDA}`).** {decisao}"
    )
)

RESULTADOS["experimento3"] = {
    "arquitetura": ESCOLHIDA,
    "plantdoc": {
        "treino": len(treino_pd),
        "validacao": len(validacao_pd),
        "teste": len(teste_pd),
        "testeOficial": TEM_TESTE_OFICIAL_PD,
    },
    "comparacao": comparacao,
    "ganhoCampoF1": ganho_campo,
    "perdaLaboratorioF1": perda_lab,
    "producao": EXPERIMENTO_PRODUCAO,
    "decisao": decisao,
    "historico": HISTORICO_EXP3,
}
salvar_json("experimento3_ajuste_fino.json", RESULTADOS["experimento3"])

## 8. Grad-CAM: o modelo olha a lesão ou o fundo?

Uma acurácia alta não garante que o modelo aprendeu a doença. No PlantVillage, cada classe foi fotografada em sessões com fundo e luz parecidos, e o modelo pode aprender esses atalhos.

O **Grad-CAM** mostra que regiões da imagem mais pesaram na decisão. Funciona assim: registramos, com *hooks* do PyTorch, as ativações da última camada convolucional e os gradientes da classe prevista em relação a elas. A média dos gradientes em cada canal diz a importância daquele canal; somamos os canais ponderados por essa importância e ficamos só com a parte positiva. O resultado é um mapa de calor do tamanho da imagem.

Para medir em números, usamos a versão **segmentada** do PlantVillage (a mesma foto com o fundo removido) e calculamos que fração da ativação cai sobre a folha.

As fotos do PlantDoc passam antes por um filtro com o CLIP que descarta imagens com pessoas, mãos ou rostos, para nenhuma imagem de pessoa ir para o repositório.

In [ ]:
# Grad-CAM com hooks
class GradCam:
    def __init__(self, modelo: nn.Module, camada: nn.Module):
        self.modelo = modelo
        self.ativacoes = None
        self.gradientes = None
        self.gancho = camada.register_forward_hook(self._guardar_ativacao)

    def _guardar_ativacao(self, modulo, entrada, saida):
        self.ativacoes = saida
        saida.register_hook(self._guardar_gradiente)

    def _guardar_gradiente(self, gradiente):
        self.gradientes = gradiente

    def mapa(self, tensor: torch.Tensor) -> tuple[np.ndarray, int]:
        self.modelo.eval()
        self.modelo.zero_grad(set_to_none=True)
        saida = self.modelo(tensor.unsqueeze(0).to(DISPOSITIVO))
        classe = int(saida.argmax(1))
        saida[0, classe].backward()
        importancia = self.gradientes.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((importancia * self.ativacoes).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=tensor.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy(), classe

    def remover(self) -> None:
        self.gancho.remove()


def mostrar_cams(
    conjunto: ImagensFolhas, posicoes: list[int], titulo: str, arquivo: str, gradcam: GradCam
) -> None:
    fig, eixos = plt.subplots(2, len(posicoes), figsize=(2.6 * len(posicoes), 5.6))
    for coluna, i in enumerate(posicoes):
        tensor, alvo = conjunto[i]
        cam, previsto = gradcam.mapa(tensor)
        imagem = desnormalizar(tensor)
        eixos[0, coluna].imshow(imagem)
        eixos[0, coluna].set_title(f"real: {ROTULOS[alvo]}", fontsize=7)
        eixos[1, coluna].imshow(imagem)
        eixos[1, coluna].imshow(cam, cmap="jet", alpha=0.45)
        cor = "green" if previsto == alvo else "red"
        eixos[1, coluna].set_title(f"previsto: {ROTULOS[previsto]}", fontsize=7, color=cor)
    for eixo in eixos.ravel():
        eixo.axis("off")
    fig.suptitle(titulo)
    salvar_figura(fig, arquivo)


for parametro in MODELO_PRODUCAO.parameters():
    parametro.requires_grad = True
gradcam = GradCam(MODELO_PRODUCAO, MODELO_PRODUCAO.features[-1])

In [ ]:
# Grad-CAM no PlantVillage e fração da ativação sobre a folha
def caminho_segmentado(caminho: str) -> str:
    _, _, classe, arquivo = caminho.split("/")
    return f"raw/segmented/{classe}/{Path(arquivo).stem}_final_masked.jpg"


recorte = transforms.Compose(
    [transforms.Resize(CONFIG["redimensionar"]), transforms.CenterCrop(CONFIG["tamanho_entrada"])]
)
amostra_cam = (
    teste_pv.sample(frac=1.0, random_state=SEED).groupby("alvo").head(1 if MODO_RAPIDO else 3)
)
baixar_do_zip([caminho_segmentado(c) for c in amostra_cam["caminho"]])

fracoes = []
for caminho, alvo in zip(amostra_cam["caminho"], amostra_cam["alvo"], strict=True):
    segmentado = arquivo_local(caminho_segmentado(caminho))
    if not segmentado.exists():
        continue
    tensor = transf_avaliacao(Image.open(arquivo_local(caminho)).convert("RGB"))
    cam, _ = gradcam.mapa(tensor)
    folha = np.asarray(recorte(Image.open(segmentado).convert("RGB"))).max(axis=2) > 20
    fracoes.append(
        {
            "rotulo": ROTULOS[alvo],
            "fracaoNaFolha": float(cam[folha].sum() / (cam.sum() + 1e-8)),
            "areaDaFolha": float(folha.mean()),
        }
    )
fracoes = pd.DataFrame(fracoes)
media_folha, media_area = fracoes["fracaoNaFolha"].mean(), fracoes["areaDaFolha"].mean()
display(
    Markdown(
        f"Em {len(fracoes)} fotos do teste do PlantVillage, **{media_folha:.1%} da ativação "
        "do Grad-CAM cai sobre a folha**, "
        f"que ocupa em média {media_area:.1%} da imagem. Se o modelo ignorasse a folha, a "
        "fração ficaria perto da área."
    )
)

posicoes_pv = (
    teste_pv.sample(frac=1.0, random_state=SEED).drop_duplicates("alvo").index[:6].tolist()
)
mostrar_cams(
    ds_teste_pv,
    posicoes_pv,
    "Grad-CAM: PlantVillage (laboratório)",
    "gradcam_plantvillage.png",
    gradcam,
)
RESULTADOS["gradcam"] = {
    "imagens": len(fracoes),
    "fracaoAtivacaoNaFolha": media_folha,
    "areaMediaDaFolha": media_area,
}

In [ ]:
# Grad-CAM no PlantDoc, sem imagens com pessoas
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval()
processador_clip = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
TEXTOS_CLIP = [
    "a photo of a plant leaf",
    "a photo of a person",
    "a photo of a human hand",
    "a photo of a human face",
]


@torch.inference_mode()
def sem_pessoas(imagem: Image.Image) -> bool:
    entradas = processador_clip(text=TEXTOS_CLIP, images=imagem, return_tensors="pt", padding=True)
    probabilidades = clip(**entradas).logits_per_image.softmax(dim=-1)[0]
    return float(probabilidades[1:].sum()) < 0.15


ds_teste_pd = ImagensFolhas(amostras_pd(teste_pd), transf_avaliacao, PLANTDOC)
posicoes_pd, descartadas = [], 0
for i in (
    teste_pd.sample(frac=1.0, random_state=SEED).drop_duplicates("alvo").index.tolist()
    + teste_pd.index.tolist()
):
    if len(posicoes_pd) == 6:
        break
    if i in posicoes_pd:
        continue
    if sem_pessoas(ds_teste_pd.imagem(i)):
        posicoes_pd.append(i)
    else:
        descartadas += 1
print(f"Imagens descartadas pelo filtro de pessoas: {descartadas}")
mostrar_cams(
    ds_teste_pd, posicoes_pd, "Grad-CAM: PlantDoc (campo)", "gradcam_plantdoc.png", gradcam
)
gradcam.remover()
RESULTADOS["gradcam"]["plantdocDescartadasPorPessoa"] = descartadas

## 9. Exportação

O serviço de IA (`app/visao`) lê dois arquivos:

- `modelos/visao.pt`: só os pesos (`state_dict`) do modelo de produção. O serviço recria a arquitetura com o torchvision, troca a última camada por uma de 17 saídas e carrega os pesos.
- `modelos/visao.json`: tudo que o serviço precisa para usar os pesos do jeito certo: arquitetura, versão dos pesos de origem, **classes na ordem de saída**, média e desvio da normalização, tamanho de entrada, temperatura da calibração e as métricas principais.

As evidências de avaliação (JSON e PNG) vão para `avaliacao/visao/`. A última célula gera um zip com a mesma estrutura de pastas do repositório.

In [ ]:
# Pesos, metadados e resumo
temperatura_producao = TEMPERATURA_EXP3 if EXPERIMENTO_PRODUCAO == "exp3" else TEMPERATURA_EXP1
logits_pv_producao = logits_pv_exp3 if EXPERIMENTO_PRODUCAO == "exp3" else logits_teste_exp1
logits_pd_producao = (
    logits_pd_exp3 if EXPERIMENTO_PRODUCAO == "exp3" else logits_pd_exp1[linhas_teste_pd]
)
prob_campo = softmax_na_cultura(logits_pd_producao, culturas_teste_pd, temperatura_producao)
pesos_base = ARQUITETURAS[ESCOLHIDA][1]

arquivo_pesos = MODELOS / "visao.pt"
torch.save({k: v.detach().cpu() for k, v in MODELO_PRODUCAO.state_dict().items()}, arquivo_pesos)
sha256 = hashlib.sha256(arquivo_pesos.read_bytes()).hexdigest()

producao = comparacao[EXPERIMENTO_PRODUCAO]
metadados = {
    "arquitetura": ESCOLHIDA,
    "pesosBase": f"{type(pesos_base).__name__}.{pesos_base.name}",
    "cabeca": f"classifier[-1] = Linear(in_features, {NUM_CLASSES})",
    "experimento": EXPERIMENTO_PRODUCAO,
    "classes": ROTULOS,
    "classesPlantVillage": CLASSES_PV,
    "culturas": CULTURAS.tolist(),
    "nomes": NOMES,
    "media": MEDIA,
    "desvio": DESVIO,
    "tamanhoEntrada": CONFIG["tamanho_entrada"],
    "redimensionar": CONFIG["redimensionar"],
    "temperatura": temperatura_producao,
    "data": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "versoes": {"torch": torch.__version__, "torchvision": torchvision.__version__},
    "sha256": sha256,
    "modoRapido": MODO_RAPIDO,
    "metricas": {
        "plantvillageTeste": {
            "acuracia": producao["plantvillage"]["acuracia"],
            "f1Macro": producao["plantvillage"]["f1Macro"],
        },
        "plantdocTesteNaCultura": {
            "acuracia": producao["plantdoc"]["naCultura"]["acuracia"],
            "f1Macro": producao["plantdoc"]["naCultura"]["f1Macro"],
        },
        "ece": {
            "antes": ece(softmax(logits_pv_producao), alvos_teste_pv),
            "depois": ece(softmax(logits_pv_producao, temperatura_producao), alvos_teste_pv),
            "plantdocNaCultura": ece(prob_campo, alvos_teste_pd),
        },
        "tamanhoMb": arquivo_pesos.stat().st_size / 2**20,
        "latenciaCpuMs": latencia_cpu_ms(MODELO_PRODUCAO),
    },
}
(MODELOS / "visao.json").write_text(
    json.dumps(metadados, ensure_ascii=False, indent=2, default=para_json), encoding="utf-8"
)
RESULTADOS["producao"] = metadados
salvar_json("resumo.json", RESULTADOS)
print(
    json.dumps(
        {
            k: v
            for k, v in metadados.items()
            if k not in ("classesPlantVillage", "nomes", "culturas")
        },
        ensure_ascii=False,
        indent=2,
        default=para_json,
    )
)

In [ ]:
# Zip para baixar
arquivo_zip = SAIDA / "valence_visao.zip"
with zipfile.ZipFile(arquivo_zip, "w", zipfile.ZIP_DEFLATED) as destino:
    for pasta in (MODELOS, SAIDA / "avaliacao"):
        for arquivo in sorted(pasta.rglob("*")):
            if arquivo.is_file():
                destino.write(arquivo, arquivo.relative_to(SAIDA).as_posix())
with zipfile.ZipFile(arquivo_zip) as conferencia:
    print("\n".join(conferencia.namelist()))
print(f"{arquivo_zip} ({arquivo_zip.stat().st_size / 2**20:.1f} MB)")
if EM_COLAB:
    from google.colab import files

    files.download(str(arquivo_zip))

## 10. Limitações observadas

O texto abaixo é montado a partir dos números desta execução.

In [ ]:
# Limitações com os números desta execução
pv = producao["plantvillage"]
campo = producao["plantdoc"]["naCultura"]
f1_campo_classes = {k: v["f1-score"] for k, v in campo["porClasse"].items()}
piores = sorted(f1_campo_classes.items(), key=lambda kv: kv[1])[:3]
ausentes_pd = sorted(set(ROTULOS) - set(mapeadas_pd["rotulo"]))
curva_06 = curva_cultura.set_index("limite").loc[0.6]

itens = [
    "**Laboratório x campo.** No teste do PlantVillage o modelo de produção tem F1 macro de "
    f"{pv['f1Macro']:.4f} e "
    f"acurácia de {pv['acuracia']:.1%}. No teste do PlantDoc, com a cultura informada, cai "
    "para F1 macro de "
    f"{campo['f1Macro']:.4f} e acurácia de {campo['acuracia']:.1%}. O número do PlantVillage "
    "não representa o uso real.",
    "**Classes mais fracas no campo:** "
    + ", ".join(f"`{r}` (F1 {v:.2f})" for r, v in piores)
    + ".",
    f"**Classes sem nenhuma foto de campo:** {', '.join(f'`{r}`' for r in ausentes_pd)}. "
    "Para elas não há medida fora do laboratório.",
    f"**Poucas fotos de campo:** o teste do PlantDoc tem {len(teste_pd)} imagens em "
    f"{teste_pd['alvo'].nunique()} classes, "
    f"de {teste_pd['alvo'].value_counts().min()} a {teste_pd['alvo'].value_counts().max()} "
    "por classe; "
    "com tão poucas imagens, uma foto a mais ou a menos muda bastante o F1 de cada classe.",
    f"**Calibração:** a temperatura {temperatura_producao:.2f} levou o ECE no PlantVillage de "
    f"{metadados['metricas']['ece']['antes']:.4f} para "
    f"{metadados['metricas']['ece']['depois']:.4f}, mas no "
    "PlantDoc, com a cultura informada, o ECE é "
    f"{metadados['metricas']['ece']['plantdocNaCultura']:.4f}. "
    "A temperatura foi ajustada com fotos parecidas com as de treino e não garante "
    "calibração no campo.",
    f"**Separação por folha incompleta:** {len(folhas_em_comum)} folhas apareciam no treino e "
    "no teste oficiais "
    f"({removidas_do_treino} imagens retiradas do treino) e {fallback:.1%} das imagens não "
    "têm `leaf_id` no mapa. "
    "Para essas, fotos da mesma folha podem estar em lados diferentes, e o resultado no "
    "PlantVillage pode ser otimista.",
    "**Limite de 0,6:** no PlantVillage, com a cultura informada, "
    f"{curva_06['cobertura']:.1%} das fotos passam do limite, "
    f"com {curva_06['acuracia']:.1%} de acerto. Em fotos de campo, mais casos devem ir para "
    "o agrônomo.",
    f"**Atalhos visuais:** {media_folha:.1%} da ativação do Grad-CAM cai sobre a folha, que "
    f"ocupa {media_area:.1%} da imagem. "
    "O resto está no fundo, que no PlantVillage é parecido dentro de cada sessão de fotos.",
    "**Vírus do enrolamento-amarelo (TYLCV):** a Embrapa registra que essa espécie ainda não "
    "foi detectada no Brasil; "
    "aqui, sintomas parecidos vêm de outros begomovírus, que o modelo nunca viu.",
    "**Uma doença por foto:** o modelo escolhe uma classe; folhas com duas doenças ou com "
    "dano de inseto e doença juntos "
    "não são representadas.",
    "**Vazamento possível no PlantDoc:** sem identificação da folha de origem, fotos "
    "parecidas podem estar no treino e no "
    "teste do Experimento 3.",
]
if MODO_RAPIDO:
    itens.insert(
        0,
        "**Execução em modo rápido:** poucas imagens e uma época por fase. Estes números só "
        "confirmam que o "
        "notebook roda; não servem como avaliação.",
    )
display(Markdown("\n".join(f"- {item}" for item in itens)))
salvar_json("limitacoes.json", {"itens": itens})